# 9. Score and Force Guidance: Drifting toward Boltzmann

Every notebook so far has taken the training set as the definition of the target. In the physical sciences that is
usually wrong. What you *want* is the Boltzmann distribution

$$p_{\text{Boltz}}(\mathbf{x}) \propto \exp\!\big(-\mathcal{E}(\mathbf{x})/k_BT\big),$$

and what you *have* is a finite set of configurations collected under enhanced sampling, at the wrong temperature,
from unconverged trajectories, or biased by whatever made the simulation affordable. A generative model trained to
match $p_{\text{data}}$ faithfully reproduces the bias.

But you also have something images never give you: **the energy function itself**, or at least its gradient. Force
labels come free from any molecular simulation and from every neural network potential. This notebook is about using
them, following *Drifting to Boltzmann* (`THEORY.md` §12).

The chain of ideas:

1. For a **Gaussian** kernel, drifting's attraction term is *exactly* a kernel-weighted average of the score of the
   sampling distribution (the **Drifting Score Identity**). We verify this and watch it converge at the Monte-Carlo
   rate — and watch the Laplace kernel fail it by 40–60%.
2. Boltzmann's score is the force over $k_BT$. Substituting gives an attraction term computable **entirely from
   forces**.
3. Two ways to use it: interpolate the *displacements* (**FI**) or reweight the *kernel* (**FK**).
4. FK with exact energies has a sharp, checkable prediction: if the data is Boltzmann at the wrong temperature, the
   right tilt is $\gamma^\star = 1 - T_{\text{target}}/T_{\text{data}}$. We test it to well under a percent.
5. **Which mechanism wins depends on the representation**, and the reason generalises far beyond molecules: drift
   modifications that change *direction* can leave the data manifold; ones that change only *weights* cannot.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. The Drifting Score Identity

For a Gaussian kernel $k(\mathbf{x},\mathbf{y})=\exp(-\|\mathbf{x}-\mathbf{y}\|^2/2\tau^2)$, the attraction term
obeys (`THEORY.md` eq. 49)

$$\mathbf{V}^{+}_{p}(\mathbf{x}) \;=\; \sum_j \bar k(\mathbf{x},\mathbf{y}_j)\,(\mathbf{y}_j-\mathbf{x})
\;=\; \tau^2\,\mathbb{E}_{p}\!\big[\bar k(\mathbf{x},\mathbf{y})\,\nabla_{\mathbf{y}}\log p(\mathbf{y})\big].$$

The proof is one integration by parts, using
$k(\mathbf{x},\mathbf{y})(\mathbf{y}-\mathbf{x}) = -\tau^2\nabla_{\mathbf{y}}k(\mathbf{x},\mathbf{y})$ — an identity
that holds *only* for the Gaussian. Two readings:

* **Left to right:** the displacement-based attraction you have been computing since Notebook 1 was secretly a
  score estimator all along.
* **Right to left:** if you know the score, you can compute the attraction *without displacements* — which is the
  whole point, because for Boltzmann the score is the force.

This is a population identity, so a finite-sample check should converge at $N^{-1/2}$. Take $p=\mathcal{N}(0,I)$
where the score is analytic.

In [ ]:
DT = torch.float64
D = 2

def score_gauss(y, sig=1.0): return -y / (sig ** 2)

def V_plus(x, y, tau, kernel="gauss"):
    d = torch.cdist(x, y)
    logits = -(d ** 2) / (2 * tau * tau) if kernel == "gauss" else -d / tau
    w = logits.softmax(-1)
    return w @ y - x, w

x = torch.randn(64, D, device=device, dtype=DT) * 0.8
print(f"{'tau':>5s} {'N':>9s} {'kernel':>9s} {'||V+||':>9s} {'rel. residual':>14s}")
for tau in [0.5, 1.0]:
    for N in [10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6]:
        y = torch.randn(N, D, device=device, dtype=DT)
        V, w = V_plus(x, y, tau, "gauss")
        rhs = tau * tau * (w @ score_gauss(y))
        print(f"{tau:5.1f} {N:9d} {'gauss':>9s} {V.norm(dim=-1).mean():9.4f} "
              f"{((V - rhs).norm() / V.norm()).item():14.4f}")
    y = torch.randn(10 ** 6, D, device=device, dtype=DT)
    V, w = V_plus(x, y, tau, "laplace")
    rhs = tau * tau * (w @ score_gauss(y))
    print(f"{tau:5.1f} {10**6:9d} {'laplace':>9s} {V.norm(dim=-1).mean():9.4f} "
          f"{((V - rhs).norm() / V.norm()).item():14.4f}  <-- identity does not hold")

The Gaussian residual falls like $N^{-1/2}$ — 0.18, 0.047, 0.016, 0.007 across three decades — so the identity is
exact and what we are watching is pure Monte-Carlo error. The Laplace residual sits at 37–57% and does not move: for
the Laplace kernel, $\nabla_{\mathbf{y}}k$ picks up an extra factor $1/\|\mathbf{y}-\mathbf{x}\|$ and the
integration by parts fails.

**This is why force-guided drifting switches to a Gaussian kernel** even though the paper prefers Laplace
for image generation (Notebook 2 §6). It is a real trade: you give up the Laplace kernel's dynamic range to gain an exact score
interpretation. Know which one you are buying.

## 2. From score to force

Boltzmann's score is the normalised force:
$\nabla\log p_{\text{Boltz}} = -\nabla\mathcal{E}/k_BT = \mathbf{F}/k_BT$. Substituting into the identity,

$$\mathbf{V}^{+}_{p_{\text{Boltz}}}(\mathbf{x}) = \tau^2\,\mathbb{E}_{p_{\text{Boltz}}}\!\left[\bar k(\mathbf{x},\mathbf{y})\,\frac{\mathbf{F}(\mathbf{y})}{k_BT}\right].$$

The test system: a 2-D double well, well separated at low temperature, easily crossed at high temperature. Data is
sampled at $T_{\text{data}}=2.5$; the target is $T_{\text{target}}=1.0$. In a real system the data would come from
overdamped Langevin dynamics; here it is sampled exactly by grid inversion (see the code comment).

In [ ]:
kB = 1.0
T_DATA, T_TARGET = 2.5, 1.0

def energy(x):  return 2.5 * (x[..., 0] ** 2 - 1) ** 2 + 0.5 * x[..., 1] ** 2
def force(x):
    return torch.stack([-10.0 * (x[..., 0] ** 2 - 1) * x[..., 0], -x[..., 1]], -1)

# check the force really is -grad E
xt = torch.randn(8, 2, device=device, requires_grad=True)
print("max |analytic F + grad E| =",
      (-torch.autograd.grad(energy(xt).sum(), xt)[0] - force(xt.detach())).abs().max().item())

# ---- exact reference distributions, by grid quadrature ----------------------------
# The physics of sampling this is Langevin dynamics, and that is how you
# would generate the data in a real system. Here we want the REFERENCE to be exact, so that any
# discrepancy we measure later belongs to the method and not to an unconverged sampler - a stiff
# quartic well is exactly the case where MALA gets sticky in the tails and biases <E> by ~10%.
# In 2-D we can simply invert the CDF on a grid.
LIMX, LIMY, NG = 4.0, 6.0, 1024
gx = torch.linspace(-LIMX, LIMX, NG, device=device)
gy = torch.linspace(-LIMY, LIMY, NG, device=device)
XX, YY = torch.meshgrid(gx, gy, indexing="ij")
EG = energy(torch.stack([XX, YY], -1))
hx, hy = gx[1] - gx[0], gy[1] - gy[0]

def exact_moments(T):
    w = torch.exp(-(EG - EG.min()) / T); w = w / w.sum()
    mx, my = (w * XX).sum(), (w * YY).sum()
    return ((w * EG).sum().item(),
            ((w * (XX - mx) ** 2).sum()).sqrt().item(),
            ((w * (YY - my) ** 2).sum()).sqrt().item())

def boltzmann_sample(n, T, seed=0):
    """Exact samples from exp(-E/kT) by multinomial grid inversion plus within-cell jitter."""
    g = torch.Generator(device=device); g.manual_seed(seed)
    w = torch.exp(-(EG - EG.min()) / T).flatten(); w = w / w.sum()
    idx = torch.multinomial(w, n, replacement=True, generator=g)
    i, j = idx // NG, idx % NG
    return torch.stack([gx[i] + (torch.rand(n, device=device, generator=g) - 0.5) * hx,
                        gy[j] + (torch.rand(n, device=device, generator=g) - 0.5) * hy], -1)

X_data = boltzmann_sample(60000, T_DATA, seed=0)
X_targ = boltzmann_sample(60000, T_TARGET, seed=1)

def report(x, tag):
    print(f"{tag:26s} <E> {energy(x).mean():7.4f}   std ({x[:,0].std():.4f}, {x[:,1].std():.4f})"
          f"   P(x1>0) {(x[:,0]>0).float().mean():.3f}")
for T, x, tag in [(T_DATA, X_data, "data"), (T_TARGET, X_targ, "target")]:
    e, sx, sy = exact_moments(T)
    print(f"{tag + f'  (T={T})':26s} <E> {e:7.4f}   std ({sx:.4f}, {sy:.4f})    <- exact")
    report(x, "   sampled")

Eplot = EG.cpu()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].contourf(XX.cpu(), YY.cpu(), Eplot, levels=30, cmap="viridis")
ax[0].set_title("energy landscape", fontsize=9)
for a, (x, t) in zip(ax[1:], [(X_data, f"data, T={T_DATA}"), (X_targ, f"target, T={T_TARGET}")]):
    a.scatter(x[:4000, 0].cpu(), x[:4000, 1].cpu(), s=1.5, alpha=0.3, lw=0)
    a.set_title(t, fontsize=9)
for a in ax: a.set_xlim(-2.5, 2.5); a.set_ylim(-3, 3); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

The hot data is broader in both coordinates and spends more time on the barrier. A model that matches it exactly is a
model of the wrong ensemble.

## 3. Force-Interpolated drifting (FI)

Displacement form and force form are two expressions for the same population quantity, so interpolate them
term-by-term (`THEORY.md` eq. 51):

$$\mathbf{V}^{+}_{\omega}(\mathbf{x}) = \sum_j \bar k(\mathbf{x},\mathbf{y}_j^{+})
\Big[(1-\omega)\big(\mathbf{y}_j^{+}-\mathbf{x}\big) + \omega\,\tau^2\tfrac{\mathbf{F}(\mathbf{y}_j^{+})}{k_BT}\Big].$$

Because the two halves agree in the population limit *when the data is already Boltzmann at $T$*, and disagree
otherwise, $\omega$ linearly scales an explicit correction toward Boltzmann:

$$\mathbf{V}^{+}_{\omega} = \mathbf{V}^{+}_{p_{\text{data}}} + \omega\,\tau^2\,\mathbb{E}_{p_{\text{data}}}
\Big[\bar k\,\nabla\log\tfrac{p_{\text{Boltz}}}{p_{\text{data}}}\Big].$$

$\omega$ is also **variance control**. In real molecular systems $\|\tau^2\mathbf{F}/k_BT\|$ can be $\sim150\times$
larger than $\|\mathbf{y}-\mathbf{x}\|$, so $\omega=1$ would swamp the field with force noise while $\omega=0.01$
supplies the same correction *direction* at 1% of the variance.

Measuring that ratio here also gives us a second, sharper check on the score identity. If the data is Boltzmann at
$T_{\text{data}}$ then $\nabla\log p_{\text{data}} = \mathbf{F}/k_BT_{\text{data}}$, so the displacement term equals
$\tau^2\mathbb{E}[\bar k\,\mathbf{F}]/k_BT_{\text{data}}$ while the force term (written with $T_{\text{target}}$)
equals $\tau^2\mathbb{E}[\bar k\,\mathbf{F}]/k_BT_{\text{target}}$. Their ratio should therefore be exactly
$T_{\text{data}}/T_{\text{target}} = 2.5$.

In [ ]:
tau_g = 0.35
xq = X_data[:512]
yp = X_data[512:2560]
w = (-torch.cdist(xq, yp).pow(2) / (2 * tau_g ** 2)).softmax(-1)
disp_term = (w @ yp - xq).norm(dim=-1).mean()
force_term = (tau_g ** 2 * (w @ force(yp)) / (kB * T_TARGET)).norm(dim=-1).mean()
print(f"||displacement term||                 = {disp_term:.4f}")
print(f"||tau^2 F / k T_target  term||        = {force_term:.4f}")
print(f"measured ratio                        = {force_term/disp_term:.2f}x")
print(f"predicted ratio  T_data / T_target    = {T_DATA/T_TARGET:.2f}x")
print(f"  (in a real molecular system the ratio reaches ~150x, which is why omega ~ 0.01)")

def drift_fi(x, y_pos, y_neg, tau, omega, T=T_TARGET):
    """Force-interpolated field. Gaussian kernel throughout (the score identity needs it)."""
    wp = (-torch.cdist(x, y_pos).pow(2) / (2 * tau * tau)).softmax(-1)
    ln = -torch.cdist(x, y_neg).pow(2) / (2 * tau * tau)
    ln = ln - torch.eye(x.shape[0], device=x.device) * 1e6 if y_neg is x else ln
    wn = ln.softmax(-1)
    V_att = (1 - omega) * (wp @ y_pos - x) + omega * tau * tau * (wp @ force(y_pos)) / (kB * T)
    V_rep = wn @ y_neg - x
    return V_att - V_rep

Now train. The generator is the usual small MLP; the only change from Notebook 1 is which field is used for the
target.

In [ ]:
def mlp(d_in, d_out, h=256, nl=3):
    L, d = [], d_in
    for _ in range(nl): L += [nn.Linear(d, h), nn.SiLU()]; d = h
    return nn.Sequential(*L, nn.Linear(d, d_out))

def train_guided(field_fn, steps=3000, N=256, P=512, lr=2e-3, eta=0.05, tau=0.35, seed=0):
    torch.manual_seed(seed)
    g = mlp(2, 2).to(device)
    opt = torch.optim.Adam(g.parameters(), lr, betas=(0.9, 0.95))
    ema = {k: v.detach().clone() for k, v in g.state_dict().items()}
    for s in range(steps):
        x = g(torch.randn(N, 2, device=device))
        y = X_data[torch.randint(0, X_data.shape[0], (P,), device=device)]
        # NOTE: no feature-scale normalisation here, unlike Notebook 3. Force guidance is NOT
        # scale-invariant: F is evaluated at physical coordinates, so if you rescale the data you
        # must un-rescale before calling force(). Silently normalising is a real bug that shows up
        # as the energy moving the WRONG way with omega. This data is already O(1), so S = 1.
        with torch.no_grad():
            V = field_fn(x, y, x, tau)
            V = V / V.pow(2).mean().sqrt().clamp_min(1e-8)
            target = x + eta * V
        loss = F.mse_loss(x, target)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
        d = min(0.995, (1 + s) / (10 + s))
        with torch.no_grad():
            for k, v in g.state_dict().items(): ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
    ge = mlp(2, 2).to(device); ge.load_state_dict(ema)
    with torch.no_grad():
        return ge(torch.randn(20000, 2, device=device))

print(f"{'omega':>8s} {'<E>':>9s} {'std x1':>9s} {'std x2':>9s} {'P(x1>0)':>9s}")
eD, sxD, syD = exact_moments(T_DATA)
eT, sxT, syT = exact_moments(T_TARGET)
print(f"{'data':>8s} {eD:9.4f} {sxD:9.4f} {syD:9.4f} {0.5:9.3f}")
fi_out = {}
for om in [0.0, 0.05, 0.1, 0.2, 0.4]:
    s_ = train_guided(lambda x, yp, yn, t, om=om: drift_fi(x, yp, yn, t, om))
    fi_out[om] = s_
    print(f"{om:8.2f} {energy(s_).mean():9.4f} {s_[:,0].std():9.4f} {s_[:,1].std():9.4f} "
          f"{(s_[:,0]>0).float().mean():9.3f}")
print(f"{'target':>8s} {eT:9.4f} {sxT:9.4f} {syT:9.4f} {0.5:9.3f}")

The measured ratio lands on the predicted $2.5$, which is the score identity confirmed a second way — this time
inside a live field computation rather than on a Gaussian toy.

$\omega=0$ reproduces the data (ordinary drifting; it is the control that tells you the training worked). Raising
$\omega$ moves the mean energy and both spreads monotonically toward the cold target. The correction does what the
algebra says it should.

**And note what happens at $\omega=0.4$.** The run diverges — the mean energy blows up by seven orders of
magnitude. This is not a bug; it is the variance cost, arriving exactly where the theory says it should. The force
term is $2.5\times$ the displacement term in magnitude *on average*, but its variance across the batch is far larger,
and once it dominates the drift-normalised field the target becomes noise. The paper's prescription of
$\omega\approx0.01$ in molecular systems, where the ratio is $\sim150\times$, follows directly: take the correction
*direction* and almost none of the variance. **If your $\omega$ sweep has a cliff, you have found the variance
limit, not a tuning failure.**

**And note the comment in `train_guided`.** Notebook 3's feature-scale normalisation, applied here without thought,
divides the coordinates by $S$ and then evaluates $\mathbf{F}$ at the *scaled* positions — a different physical
point. The symptom is subtle and misleading: the energy moves in the wrong direction as $\omega$ increases, which
reads as "force guidance does not work" rather than as a units bug. Force guidance is not scale-invariant, and this
is the first thing to check if your $\omega$ sweep comes out backwards.

Note what $\omega$ does **not** do: it does not reach the target exactly, at any value. §5 explains why — the
realised equilibrium sits *between* $p_{\text{data}}$ and $p_{\text{Boltz}}$ because attraction and repulsion
normalise over different measures and no longer cancel. FI is a *shift*, not a substitution.

## 4. Force-Aligned Kernel (FK), and a prediction to check

The other option leaves displacements alone and reweights the kernel toward Boltzmann-likely data
(`THEORY.md` eq. 54):

$$\bar k_{\text{corrected}}(\mathbf{x},\mathbf{y}_j) \propto k(\mathbf{x},\mathbf{y}_j)\,p_{\text{Boltz}}(\mathbf{y}_j)^{\gamma}
\quad\Longrightarrow\quad
\ell_j = -\frac{\|\mathbf{x}-\mathbf{y}_j\|^2}{2\tau^2} - \frac{\gamma\,\mathcal{E}(\mathbf{y}_j)}{k_BT}.$$

The intractable $\log Z$ cancels inside the softmax, so with energy labels this is *exact*. The effective sampling
measure becomes $p_{\text{data}}\cdot p_{\text{Boltz}}^{\gamma}/Z$, which gives a **falsifiable prediction**: if the
data is itself Boltzmann at the wrong temperature $T_{\text{data}}$, then

$$\gamma^\star = k_BT_{\text{target}}\left(\frac{1}{k_BT_{\text{target}}} - \frac{1}{k_BT_{\text{data}}}\right) = 1-\frac{T_{\text{target}}}{T_{\text{data}}} = 1-\frac{1.0}{2.5} = 0.6.$$

The cleanest test does not need a network at all: the tilt is a reweighting, so apply it as importance weights to the
data and measure the reweighted ensemble directly.

In [ ]:
E_data = energy(X_data)
gam_star = 1 - T_TARGET / T_DATA                     # THEORY.md eq. 56
print(f"predicted gamma* = {gam_star:.3f}\n")
print(f"{'gamma':>7s} {'<E>_w':>9s} {'std x1':>9s} {'std x2':>9s} {'ESS':>8s}")
for gam in [0.0, 0.2, 0.6, 1.0]:
    w = torch.softmax(-gam * E_data / (kB * T_TARGET), 0)
    mE = (w * E_data).sum()
    mu = (w[:, None] * X_data).sum(0)
    sd = ((w[:, None] * (X_data - mu) ** 2).sum(0)).sqrt()
    print(f"{gam:7.2f} {mE:9.4f} {sd[0]:9.4f} {sd[1]:9.4f} {1/(w**2).sum():8.0f}")
eT, sxT, syT = exact_moments(T_TARGET)
print(f"{'target':>7s} {eT:9.4f} {sxT:9.4f} {syT:9.4f}      exact")
w = torch.softmax(-gam_star * E_data / (kB * T_TARGET), 0)
mu = (w[:, None] * X_data).sum(0); sd = ((w[:, None] * (X_data - mu) ** 2).sum(0)).sqrt()
print(f"\nat gamma = gamma* = {gam_star:.1f}:")
print(f"   mean-energy error {abs((w*E_data).sum().item()-eT)/eT:.2%}")
print(f"   std x1 error      {abs(sd[0].item()-sxT)/sxT:.2%}")
print(f"   std x2 error      {abs(sd[1].item()-syT)/syT:.2%}")

The prediction holds to well under a percent in every moment: at $\gamma=0.6$ the reweighted ensemble reproduces the
cold target's mean energy and both per-coordinate spreads, while $\gamma=0.2$ under-corrects and $\gamma=1.0$
over-corrects. This is a genuine quantitative test of the theory, not a plausibility check — the number 0.6 came out
of the algebra before the measurement, and this level of agreement is only visible because the reference was computed
exactly rather than sampled.

Two practical notes. **ESS falls as $\gamma$ rises** (here 60k → 30k): the tilt discards data, and in a real system
with a rugged landscape it can discard almost all of it. And $\gamma$ enters the *kernel*, so the tilt competes with
the distance term: at large $\tau$ the energy dominates and the field stops being local.

### FK from forces alone

Energies are often unavailable while forces are not (a neural network potential typically outputs forces; absolute
energies may be arbitrary or per-system offset). Taylor-expand around the query,
$\mathcal{E}(\mathbf{y}_j) \approx \mathcal{E}(\mathbf{x}) - \mathbf{F}(\mathbf{y}_j)\cdot(\mathbf{y}_j-\mathbf{x})$;
the $\mathcal{E}(\mathbf{x})$ term is constant in $j$ and cancels in the softmax:

$$\ell_j \approx -\frac{\|\mathbf{x}-\mathbf{y}_j\|^2}{2\tau^2} + \gamma\,\frac{\mathbf{F}(\mathbf{y}_j)\cdot(\mathbf{y}_j-\mathbf{x})}{k_BT}.$$

Geometrically: a data point whose force points *away* from the query — so that travelling out to it goes downhill —
gets upweighted. The error is controlled by $\varepsilon = \tau\|\nabla^2\mathcal{E}\|/\|\nabla\mathcal{E}\|$, so it
is exact for locally affine potentials and degrades as $\tau$ grows.

In [ ]:
def fk_weights(x, y, tau, gam, mode="exact", T=T_TARGET):
    base = -torch.cdist(x, y).pow(2) / (2 * tau * tau)
    if gam == 0.0:  return base.softmax(-1)
    if mode == "exact":
        return (base - gam * energy(y)[None] / (kB * T)).softmax(-1)
    tilt = gam * ((force(y)[None] * (y[None] - x[:, None])).sum(-1)) / (kB * T)   # eq. 55
    return (base + tilt).softmax(-1)

xq = X_data[:1024]
yp = X_data[1024:5120]
print(f"{'tau':>6s} {'gamma':>7s} {'cos(exact, taylor)':>20s} {'||dV||/||V||':>14s} {'epsilon':>9s}")
for tau in [0.05, 0.1, 0.2, 0.4, 0.8, 1.6]:
    for gam in [0.6]:
        we = fk_weights(xq, yp, tau, gam, "exact")
        wt = fk_weights(xq, yp, tau, gam, "taylor")
        Ve, Vt = we @ yp - xq, wt @ yp - xq
        cos = F.cosine_similarity(Ve, Vt, dim=-1).mean()
        # epsilon = tau * ||Hess|| / ||grad||, sampled on the data
        H = torch.stack([30 * yp[:, 0] ** 2 - 10, torch.ones_like(yp[:, 0])], -1).abs().max(-1).values
        eps = (tau * H / force(yp).norm(dim=-1).clamp_min(1e-6)).median()
        print(f"{tau:6.2f} {gam:7.2f} {cos:20.4f} {((Ve-Vt).norm()/Ve.norm()):14.4f} {eps:9.2f}")

At small $\tau$ the force-only approximation is nearly perfect ($\cos>0.98$); as $\tau$ grows past the curvature
scale ($\varepsilon\sim1$) it degrades, exactly as the error bound predicts. The non-monotone tail at the largest
$\tau$ is not a contradiction: once the kernel is so wide that both variants weight nearly every sample equally,
they agree again — on a field that has stopped being local and stopped being useful. Read the $\varepsilon$ column,
not the cosine, when deciding whether the approximation is safe.

If you only have forces, keep $\tau$ below the scale on which the potential's curvature changes — which is also,
usefully, the regime where drifting's kernel is doing its most informative work.

## 5. Where the equilibrium actually lands

Force alignment **deliberately breaks anti-symmetry**: the attraction now uses a different kernel (or different
displacements) from the repulsion. Notebook 1 §9 showed that breaking anti-symmetry by rescaling
$\mathbf{V}^{+}$ against $\mathbf{V}^{-}$ collapses the model. Why is this different?

Because *what* is broken differs. Rescaling changes the balance between attraction and repulsion, so the equilibrium
condition $\mathbf{V}=\mathbf{0}$ becomes "attraction is $c\times$ repulsion", which has no fixed point at any
sensible distribution — the model shrinks or explodes. Force alignment leaves the displacements untouched and changes
only *which positives are weighted how*, which moves the equilibrium to a different distribution rather than
destroying it. The three cases (`THEORY.md` §12.5):

* $q = p_{\text{data}}$ is **no longer** an equilibrium when $p_{\text{data}}\neq p_{\text{Boltz}}$: the
  anti-symmetric part cancels, the correction survives, and it pushes past the data.
* $q = p_{\text{Boltz}}$ is an exact equilibrium **iff** $p_{\text{data}}=p_{\text{Boltz}}$ — i.e. only in the case
  where you did not need the correction.
* Otherwise the realised equilibrium is **between** them, moved toward Boltzmann by $\omega$ (or $\gamma$).

That is exactly what §3's table shows, and it is worth internalising: force guidance is a **controlled bias**, not a
correction that converges to the truth. You are trading data fidelity for physical fidelity, on a dial. The
honest way to report it is with both numbers — distance to the data ensemble *and* distance to the Boltzmann
ensemble — never one alone.

In [ ]:
# the dial, plotted: how far each model sits from data vs from target
def moments(x): return torch.tensor([energy(x).mean(), x[:, 0].std(), x[:, 1].std()], device=device)
m_data, m_targ = moments(X_data), moments(X_targ)
def rel_gap(x, m): return ((moments(x) - m) / m).abs().mean().item()

oms = sorted(fi_out)
plt.figure(figsize=(6, 4))
plt.plot(oms, [rel_gap(fi_out[o], m_data) for o in oms], "o-", label="distance to the DATA ensemble")
plt.plot(oms, [rel_gap(fi_out[o], m_targ) for o in oms], "s-", label="distance to the BOLTZMANN target")
plt.xlabel(r"force-interpolation weight $\omega$"); plt.ylabel("mean relative moment error")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()

Two curves crossing: that is the trade-off made visible. There is no value of $\omega$ at which both are zero,
because the data is not the target.

## 6. The representation reversal

Here is the finding with no analogue in image generation: **which mechanism works reverses with the representation.**

* In **Cartesian coordinates**, FI dominates. Forces *are* displacement directions; interpolating them with data
  displacements is dimensionally and physically natural.
* In **pairwise-distance space**, FK dominates. A "force" in distance space is an abstract vector, and adding it to a
  distance vector produces something that is **not the distance vector of any 3-D structure** — a manifold violation.
  Reweighting the kernel leaves the displacements alone; since they are differences of *real* geometries, bond
  constraints survive.

The general principle:

> **Any modification that changes the drift's *direction* risks leaving the data manifold. Modifications that change
> only the *weights* cannot.** Weight-space guidance is the safer default whenever the output space is constrained.

Let us measure it. A 6-atom chain with fixed bond lengths and angles; conformers generated by random dihedrals. The
manifold is the set of distance vectors realisable by 3-D points — testable by classical MDS: form the Gram matrix
$B = -\tfrac12 J\,D^{(2)}J$ and ask what fraction of its spectral mass lies outside the top three eigenvalues.

In [ ]:
NA, BOND, ANGLE = 6, 1.53, math.radians(112.0)

def build_chain(dihedrals):
    """NeRF chain builder. dihedrals: [B, NA-3] -> coordinates [B, NA, 3]"""
    B = dihedrals.shape[0]
    X = torch.zeros(B, NA, 3, device=device)
    X[:, 1, 0] = BOND
    X[:, 2, 0] = BOND - BOND * math.cos(ANGLE); X[:, 2, 1] = BOND * math.sin(ANGLE)
    for i in range(3, NA):
        a, b, c = X[:, i - 3], X[:, i - 2], X[:, i - 1]
        bc = F.normalize(c - b, dim=-1)
        n = F.normalize(torch.cross(b - a, bc, dim=-1), dim=-1)
        m = torch.cross(n, bc, dim=-1)
        d = dihedrals[:, i - 3]
        local = torch.stack([-BOND * math.cos(ANGLE) * torch.ones_like(d),
                             BOND * math.sin(ANGLE) * torch.cos(d),
                             BOND * math.sin(ANGLE) * torch.sin(d)], -1)
        X[:, i] = c + (torch.stack([bc, m, n], -1) @ local.unsqueeze(-1)).squeeze(-1)
    return X

IU = torch.triu_indices(NA, NA, offset=1).to(device)
BONDED = ((IU[1] - IU[0]) == 1)                       # the 5 bonded pairs

def phi(X): return (X[..., IU[0], :] - X[..., IU[1], :]).norm(dim=-1)

def to_matrix(p):
    Dm = torch.zeros(p.shape[0], NA, NA, device=device)
    Dm[:, IU[0], IU[1]] = p; Dm[:, IU[1], IU[0]] = p
    return Dm

def embed_residual(p):
    """Fraction of spectral mass outside a rank-3 Euclidean embedding. 0 = a real 3-D structure."""
    Dm = to_matrix(p)
    J = torch.eye(NA, device=device) - 1.0 / NA
    G = -0.5 * J @ (Dm ** 2) @ J
    ev = torch.linalg.eigvalsh(G)
    pos = ev.clamp_min(0)
    return 1 - pos[:, -3:].sum(-1) / (pos.sum(-1) + ev.clamp_max(0).abs().sum(-1))

X_conf = build_chain(torch.rand(4096, NA - 3, device=device) * 2 * math.pi)
P_conf = phi(X_conf)
def bond_error(p): return (p[:, BONDED] - BOND).abs().mean()

print(f"real conformers: embedding residual {embed_residual(P_conf).mean():.2e}  "
      f"bond-length error {bond_error(P_conf):.2e} A")

# an energy defined directly on distance space (so 'forces' live there too)
def E_phi(p):  return 0.5 * ((p - 2.6) ** 2).sum(-1)
def F_phi(p):  return -(p - 2.6)

tau_p, eta_p = 0.5, 1.0
xq_p, yp_p = P_conf[:512], P_conf[512:2560]
base = -torch.cdist(xq_p, yp_p).pow(2) / (2 * tau_p ** 2)

print(f"\n{'mechanism':22s} {'embed residual':>15s} {'bond-length error (A)':>23s}")
for om in [0.0, 0.1, 0.5, 1.0]:
    w = base.softmax(-1)
    V = (1 - om) * (w @ yp_p - xq_p) + om * tau_p ** 2 * (w @ F_phi(yp_p))
    t = xq_p + eta_p * V
    print(f"FI  omega={om:<12.2f} {embed_residual(t).mean():15.5f} {bond_error(t):23.5f}")
for gam in [0.0, 0.05, 0.2]:
    w = (base - gam * E_phi(yp_p)[None]).softmax(-1)
    t = xq_p + eta_p * (w @ yp_p - xq_p)
    print(f"FK  gamma={gam:<12.2f} {embed_residual(t).mean():15.5f} {bond_error(t):23.5f}")

Read the two columns together.

* Real conformers have embedding residual $\sim10^{-8}$ and zero bond-length error: they are exactly 3-D structures
  with exactly the right bonds.
* **Even at $\omega=0$** — plain drifting in distance space — the target has a residual of ~0.024. A kernel-weighted
  *average* of distance vectors is not itself a realisable distance vector. This is a fact about distance space that
  has nothing to do with forces, and it is worth knowing before you choose that representation.
* **FI makes it about three times worse** as $\omega\to1$, and it is the only mechanism that perturbs bond lengths,
  because the force in distance space has a component along every pair — including the bonded ones that no real
  motion of the molecule can stretch.
* **FK barely moves either number.** Reweighting cannot leave the convex structure of the displacements it is
  averaging.

That is the reversal, measured. In Cartesian space the same experiment comes out the other way, because there the
force *is* a displacement and adding it moves you along the manifold rather than off it.

### The per-molecule validity crisis

The other empirical warning from that work: **aggregate statistics can look perfect while individual samples are
catastrophically wrong.** A distance-histogram comparison averages over molecules and pairs; a single sample with a
broken ring or an impossible bond contributes a few counts and vanishes.

In [ ]:
# a deliberately mixed population: 90% good structures, 10% badly corrupted ones
good = P_conf[:3000]
bad = P_conf[3000:3300].clone()
bad[:, ~BONDED] = bad[:, ~BONDED] * 1.6 + 0.4 * torch.randn_like(bad[:, ~BONDED])
mixed = torch.cat([good, bad])

def hist_dist(a, b, bins=60, lo=1.0, hi=7.0):
    ha = torch.histc(a.flatten().cpu(), bins, lo, hi); ha = ha / ha.sum()
    hb = torch.histc(b.flatten().cpu(), bins, lo, hi); hb = hb / hb.sum()
    return 0.5 * (ha - hb).abs().sum().item()

ref = P_conf[3300:]
print(f"aggregate distance-histogram TV, clean population vs reference : {hist_dist(good, ref):.4f}")
print(f"aggregate distance-histogram TV, 10%-corrupted vs reference    : {hist_dist(mixed, ref):.4f}")
print()
for name, p in [("clean", good), ("10% corrupted", mixed)]:
    r = embed_residual(p)
    print(f"{name:16s}  mean embed residual {r.mean():.4f}   "
          f"fraction of samples with residual > 0.05: {(r > 0.05).float().mean():.1%}")

The aggregate histogram moves from 0.013 to 0.035 — a change you would plausibly write off as sampling noise or a
slightly different protocol. The per-sample validity rate reports 9.1% invalid, which is not ambiguous at all. **Always
report per-sample validity**, and report it as a *fraction of samples that pass*, not as a mean — the mean of a
bimodal validity distribution describes nothing.

For structure prediction the equivalents are per-complex clash counts, per-residue bond-geometry violations, and
chirality checks; not the aggregate lDDT.

## 7. What to take from this

1. **The Gaussian kernel makes drifting a score estimator.** Exactly, to Monte-Carlo error. The Laplace kernel does
   not — choose deliberately.
2. **Forces are a free training signal** wherever you have a potential, and they enter drifting through the
   attraction term only.
3. **Two mechanisms, two failure modes.** FI changes directions and can leave the manifold; FK changes weights and
   cannot, but throws away effective sample size.
4. **Force guidance is a controlled bias, not a correction.** Report distance to the data *and* to the target.
5. **$\gamma^\star = 1 - T_{\text{target}}/T_{\text{data}}$** is a real, checkable prediction — verified to
   under 1% here. When a theory gives you a number, test the number, and make the reference exact so that the test
   measures the theory rather than your sampler.
6. **The right mechanism depends on the representation.** Weight-space guidance is the safe default in constrained
   output spaces. This is the single most transferable idea in the notebook, and it applies directly to structure
   prediction, where valid outputs are a thin manifold in $\mathbb{R}^{3N}$.
7. **Per-sample validity, always.** Aggregate metrics hide exactly the failures that matter.

## Summary

* Drifting Score Identity verified to $N^{-1/2}$ for the Gaussian kernel; Laplace fails it by ~40–60%.
* FI shifts the ensemble toward Boltzmann in proportion to $\omega$, at the cost of variance ($\tau^2F/k_BT$ is
  several times the displacement scale here, ~150$\times$ in real systems).
* FK with exact energies is a temperature reweighting: $\gamma^\star=0.6$ predicted and confirmed to well under 1%
  in mean energy and in both spreads, against an exactly-computed reference. The force-only Taylor form is near-exact while $\tau\|\nabla^2\mathcal{E}\|/\|\nabla\mathcal{E}\|\ll1$.
* The equilibrium under force guidance lies strictly between data and Boltzmann. It is a dial, not a fix.
* In distance space, FI degrades geometric validity ~3$\times$ while FK does not — the direction-vs-weight principle.
* Aggregate distance histograms hid a 10% corrupted population that per-sample validity caught immediately.

Next: [10. Discrete Drifting for Peptides](<10. Discrete Drifting for Peptides.ipynb>) — objective-steering rather
than distribution-matching, and what happens when repulsion points at something other than your own samples.